In [ ]:
import os
import sys

# Some environment variable management to get Pytoch to work inside a docker container
os.environ['USER'] = 'root'
os.environ['HOME'] = '/tmp'
os.environ['TORCH_HOME'] = '/tmp/torch'
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'

import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import datetime as dt
import numpy as np
import random
import torch

from model import AlphaZeroNet, run_neural_net
from trainer import complete_training_loop
from environment import RealGoGame

def set_seed(seed: int = 48):
    """
    Sets the random seed for reproducibility across Python and NumPy.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    print(f"Random seed set to {seed}")

def inspect_empty_board(params):
	game = RealGoGame(board_size=3)
	
	# Get raw network prediction (No MCTS)
	policy, value = run_neural_net(game, params)
	
	# 1. Check Value (Who does it think is winning?)
	# Range: -1 (White wins) to +1 (Black wins)
	print(f"Network Evaluation of Empty Board: {value:.4f}")
	if value > 0.5:
		print(">> VERDICT: It correctly knows Black has the advantage.")
	else:
		print(">> VERDICT: It is still unsure (or thinks White is winning).")

	# 2. Check Policy (Where does it want to play?)
	# Reshape the first 9 elements to 3x3 (ignoring the 'pass' move at index 9)
	policy_grid = policy[:9].reshape(3, 3)
	
	print("\nPolicy Probability Map:")
	print(np.round(policy_grid, 3))
	
	# Visual Heatmap
	plt.figure(figsize=(4, 3))
	sns.heatmap(policy_grid, annot=True, cmap="Blues", fmt=".2f")
	plt.title("Where does the AI want to play?")
	plt.show()
	
	# Check if Center is the max
	flat_idx = np.argmax(policy[:9])
	if flat_idx == 4: # Center is index 4 (1,1)
		print(">> SUCCESS: The AI prefers the Center (1,1)!")
	else:
		print(f">> FAIL: The AI prefers index {flat_idx} (Row {flat_idx//3}, Col {flat_idx%3})")

def play_vs_random(params, ai_plays_black=True):
	game = RealGoGame(board_size=3)
	print("--- New Game: AI (Black) vs Random (White) ---" if ai_plays_black else "--- New Game: Random (Black) vs AI (White) ---")
	
	moves = 0
	while not game.is_game_over() and moves < 20:
		# Determine who is playing
		is_ai_turn = (game.current_player == 1 and ai_plays_black) or \
					 (game.current_player == -1 and not ai_plays_black)
		
		if is_ai_turn:
			policy, _ = run_neural_net(game, params)
			valid_moves = game.get_valid_moves()
			best_score = -1
			best_move = None
			
			for move in valid_moves:
				if move == "PASS": idx = 9
				else: idx = move[0]*3 + move[1]
				
				if policy[idx] > best_score:
					best_score = policy[idx]
					best_move = move
			
			action = best_move
			p_type = "AI Black"
		else:
			valid_moves = game.get_valid_moves()
			action = valid_moves[np.random.randint(len(valid_moves))]
			p_type = "RND White"
			
		print(f"Move {moves+1} ({p_type}): {action}")
		game.step(action)
		print(game.board)
		moves += 1
		
	winner = game.get_reward()
	if winner == 1: print("Result: Black Wins")
	elif winner == -1: print("Result: White Wins")
	else: print("Result: Draw")

In [ ]:
tic = dt.datetime.today()
print('Cell Start', tic)

# Train the model (self play)
set_seed(48)
model = AlphaZeroNet(board_size=3, filters=16)
complete_training_loop(model, num_games=500) 

In [ ]:
inspect_empty_board(model)

In [ ]:
play_vs_random(model, ai_plays_black=True)